In [12]:
import numpy as np
import pandas as pd
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

from sklearn.model_selection import KFold
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import r2_score
import xgboost as xgb


PATH = Path('/content/drive/MyDrive/ETH/AML/proj1/data/')

X_train_dataframe = pd.read_csv(PATH / 'X_train.csv')
y_train_dataframe = pd.read_csv(PATH / 'y_train.csv')
X_test_dataframe  = pd.read_csv(PATH / 'X_test.csv')

test_ids     = X_test_dataframe['id'].astype(int).values
feature_cols = [col for col in X_train_dataframe.columns if col != 'id']

X_train_matrix = X_train_dataframe[feature_cols].values.astype(float)
y_train_vector = y_train_dataframe['y'].values.astype(float)
X_test_matrix  = X_test_dataframe[feature_cols].values.astype(float)


class CorrelationFilter:
    def __init__(self, threshold=0.95):
        self.threshold = threshold

    def fit(self, X, y=None):
        correlation_matrix = np.abs(np.nan_to_num(
            np.corrcoef(X, rowvar=False), nan=0.0, posinf=0.0, neginf=0.0
        ))
        upper_triangle = np.triu(correlation_matrix, k=1)
        self.keep_indices = ~(upper_triangle > self.threshold).any(axis=0)
        if self.keep_indices.sum() == 0:
            self.keep_indices[0] = True
        return self

    def transform(self, X):
        return X[:, self.keep_indices]


def build_xgboost_model():
    return xgb.XGBRegressor(
        n_estimators=1500,
        learning_rate=0.03,
        max_depth=6,
        min_child_weight=5,
        reg_alpha=0.1,
        reg_lambda=1.0,
        subsample=0.8,
        colsample_bytree=0.7,
        tree_method='hist',
        n_jobs=2,
        random_state=0,
        verbosity=0,
    )


def preprocess(train_X, test_X):
    imputer = SimpleImputer(strategy='median')
    train_X = imputer.fit_transform(train_X)
    test_X  = imputer.transform(test_X)

    variance_filter = VarianceThreshold(threshold=1e-8)
    train_X = variance_filter.fit_transform(train_X)
    test_X  = variance_filter.transform(test_X)

    corr_filter = CorrelationFilter(threshold=0.95)
    train_X = corr_filter.fit(train_X).transform(train_X)
    test_X  = corr_filter.transform(test_X)

    return train_X, test_X, int(corr_filter.keep_indices.sum())


kfold = KFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []

for train_idx, val_idx in kfold.split(X_train_matrix):
    fold_train_X_pp, fold_val_X_pp, _ = preprocess(
        X_train_matrix[train_idx], X_train_matrix[val_idx]
    )
    model = build_xgboost_model()
    model.fit(fold_train_X_pp, y_train_vector[train_idx])
    fold_predictions = model.predict(fold_val_X_pp)
    fold_scores.append(r2_score(y_train_vector[val_idx], fold_predictions))


X_train_pp, X_test_pp, n_features_final = preprocess(X_train_matrix, X_test_matrix)

final_model = build_xgboost_model()
final_model.fit(X_train_pp, y_train_vector)
test_predictions = final_model.predict(X_test_pp)

submission = pd.DataFrame({'id': test_ids, 'y': test_predictions})
output_path = PATH / 'y_test_pred_xgb_corrfilter.csv'
submission.to_csv(output_path, index=False)

print(f"CV R^2: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
print(f"Features kept: {n_features_final} / 832")
print(f"Saved: {output_path}")

CV R²          : 0.5025 ± 0.0565
Features kept  : 826 / 832
Saved          : /content/drive/MyDrive/ETH/AML/proj1/data/y_test_pred_xgb_corrfilter.csv
